# RG3 가스/정상 밀도비 사례비교 앙상블
소수 가스 분포와 정상 분포의 대비.

In [1]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from scipy.special import expit
from sklearn.neighbors import KernelDensity
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
features=stdcols+['Max_Injection_Pressure','Average_Back_Pressure']
A=d[features].to_numpy(float);densitycache={}
def density_predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in densitycache:return densitycache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:return np.full((len(q),3),float(y[ref][0]) if len(ref) else 0.)
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    tr=imp.fit_transform(A[ref]);te=imp.transform(A[q])
    scale=RobustScaler().fit(tr[y[ref]==0]);tr=scale.transform(tr);te=scale.transform(te)
    out=[]
    for bandwidth in [.5,1.,2.]:
        normal=KernelDensity(bandwidth=bandwidth).fit(tr[y[ref]==0])
        gas=KernelDensity(bandwidth=bandwidth).fit(tr[y[ref]==1])
        out.append(expit(gas.score_samples(te)-normal.score_samples(te)))
    densitycache[key]=np.column_stack(out)
    return densitycache[key]
records=[];folds=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=density_predict(ref,q);inside=np.full((len(ref),3),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inside[mask]=density_predict(ref[~mask],ref[mask])
        candidates={'density3_mean':(outer.mean(axis=1),inside.mean(axis=1)),
'density3_consensus':(np.median(outer,axis=1),np.median(inside,axis=1))}
        for k,b in enumerate([.5,1.,2.]):candidates['density_bw'+str(b)]=(outer[:,k],inside[:,k])
        for name,(score,cal) in candidates.items():
            normal=cal[(y[ref]==0)&np.isfinite(cal)]
            for rate in [.005,.01,.02]:
                cut=float(np.quantile(normal,1-rate,method='higher')) if len(normal) else 1.
                for i,z in enumerate(q):records.append(dict(scheme=scheme,model=name,budget=rate,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[i]),prediction=int(score[i]>cut)))
        folds.append(dict(scheme=scheme,block=block,train=len(ref),train_gas=int(y[ref].sum()),test=len(q),test_gas=int(y[q].sum())))
        print('Completed density contrast',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name,budget),g in pp.groupby(['scheme','model','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,budget=budget,AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_density_contrast_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_density_contrast_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(folds).to_csv(tables/'rg3_density_contrast_folds.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.budget.eq(.01)].sort_values(['scheme','F1','AP'],ascending=[True,False,False])
lines=['# RG3 소수 가스 사례의 분포 대비: 밀도비 앙상블','',
'RF가중치변경이나정상영역이탈과다른접근이다. 학습정상과학습가스의조건부밀도를별도로KernelDensity로학습하고gas/normal밀도비를점수로사용한다. 라벨을사용하는지도/사례비교방식이다. 정상만학습한모델이아니다.',
'기존흔들림4개+현재사출압력/배압총6개. imputer는학습집합,RobustScaler는학습정상으로만적합한다. 대역폭0.5/1/2 세밀도모델의동일평균및중앙값앙상블고정. 균등사전확률의점수라실제가스발생확률로해석하지않는다.',
'바깥평가구간전체제외,안쪽평가점수도해당안쪽구간제외. 임계값은안쪽검증정상상위0.5/1/2%경계. 외부정답으로대역폭/기준선택하지않는다. 가스표본이없는안쪽학습은전부0점수폴백을쓰며이값이기준에영향줄수있다.',
'구간제외정상566가스17/과거정상427가스13. 소수사례의분포추정은불안정할수있다. 동일자료반복탐색으로독립최종성능아님. 실제Run이아닌관측구간/원본과거이력조건유지.',
'','| 평가 | 방식 | AP | TP | FP | FN | TN | F1 |','|---|---|---:|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} |')
(root/'reports/rg3_density_contrast_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False))


             scheme              model  budget       AP  TP  FP  FN  TN       F1
leave_one_block_out      density_bw2.0    0.01 0.094481   2  10  15 556 0.137931
leave_one_block_out      density_bw1.0    0.01 0.071604   2  17  15 549 0.111111
leave_one_block_out density3_consensus    0.01 0.049037   0  14  17 552 0.000000
leave_one_block_out      density3_mean    0.01 0.044811   0  13  17 553 0.000000
leave_one_block_out      density_bw0.5    0.01 0.037128   0   4  17 562 0.000000
   past_blocks_only      density_bw2.0    0.01 0.130238   2  10  11 417 0.160000
   past_blocks_only      density_bw1.0    0.01 0.088502   3  26  10 401 0.142857
   past_blocks_only density3_consensus    0.01 0.065977   1  21  12 406 0.057143
   past_blocks_only      density3_mean    0.01 0.057061   0  18  13 409 0.000000
   past_blocks_only      density_bw0.5    0.01 0.044447   0   5  13 422 0.000000

In [1]:
# Post-hoc diagnostic: bandwidth2 chosen after inspecting all density candidates.
a=pp.loc[pp.model.eq('density_bw2.0')&pp.budget.eq(.01)]
old=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
old=old.loc[old.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
z=a.merge(old,on=['scheme','TimeStamp'],validate='one_to_one')
z['union']=z.prediction.eq(1)|z.predicted_train_fpr1
rr=[]
for scheme,g in z.groupby('scheme'):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.union,labels=[0,1]).ravel()
    rr.append(dict(scheme=scheme,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),post_hoc=True))
union_metrics=pd.DataFrame(rr)
union_metrics.to_csv(tables/'rg3_density_contrast_post_hoc_union.csv',index=False,encoding='utf-8-sig')
print(union_metrics.to_string(index=False))


             scheme  TP  FP  FN  TN       F1  post_hoc
leave_one_block_out   5  14  12 552 0.277778      True
   past_blocks_only   3  12  10 415 0.214286      True